<a href="https://colab.research.google.com/github/Le2se0hy/FA_ProAn/blob/main/2022%EC%84%9C%EC%9A%B8%EC%95%84%ED%8C%8C%ED%8A%B8%EC%8B%A4%EA%B1%B0%EB%9E%98%EA%B0%92%EB%8D%B0%EC%9D%B4%ED%84%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
!pip -q install xmltodict tqdm pandas


In [23]:
import requests, xmltodict, pandas as pd, time

SERVICE_KEY = "0cab0ce1ca32028cbefd317ec961af3f4b5fd06d0ab7aeb3ee5f66c6c3da5f99" # 인증키

URL = "http://apis.data.go.kr/1613000/RTMSDataSvcAptTrade/getRTMSDataSvcAptTrade"

SEOUL_SGG = {
    "강남구":"11680","강동구":"11740","강북구":"11305","강서구":"11500","관악구":"11620",
    "광진구":"11215","구로구":"11530","금천구":"11545","노원구":"11350","도봉구":"11320",
    "동대문구":"11230","동작구":"11590","마포구":"11440","서대문구":"11410","서초구":"11650",
    "성동구":"11200","성북구":"11290","송파구":"11710","양천구":"11470","영등포구":"11560",
    "용산구":"11170","은평구":"11380","종로구":"11110","중구":"11140","중랑구":"11260"
}

months = [f"2022{m:02d}" for m in range(1, 13)]

def fetch_all_pages(lawd_cd: str, deal_ymd: str, num_rows=1000):
    rows = []
    page = 1

    while True:
        params = {
            "serviceKey": SERVICE_KEY,
            "LAWD_CD": lawd_cd,
            "DEAL_YMD": deal_ymd,
            "pageNo": page,
            "numOfRows": num_rows
        }

        r = requests.get(URL, params=params, headers={"User-Agent":"Mozilla/5.0"}, timeout=30)
        r.raise_for_status()

        data = xmltodict.parse(r.text)
        body = data.get("response", {}).get("body", {})
        items = body.get("items", {}).get("item", None)

        if items is None:
            break

        if isinstance(items, dict):
            rows.append(items)
        else:
            rows.extend(items)

        total = int(body.get("totalCount", 0))
        if page * num_rows >= total:
            break

        page += 1
        time.sleep(0.05)

    return rows

all_rows = []
fail_log = []

for gu, code in SEOUL_SGG.items():
    for ym in months:
      try:
        rows = fetch_all_pages(code, ym)
        for it in rows:
          it["구"] = gu
          it["계약년월"] = ym
          all_rows.extend(rows)
          time.sleep(0.05)
      except Exception as e:
        fail_log.append((gu, ym, str(e)))

df = pd.DataFrame(all_rows)

print("총 거래 건수:", len(df))
print("컬럼 수:", len(df.columns))
print("실패 건:", len(fail_log))
if fail_log:
    print("실패 예시:", fail_log[:5])

# 저장
df.to_csv("seoul_apt_trade_2022_opendata.csv", index=False, encoding="utf-8-sig")
df.to_parquet("seoul_apt_trade_2022_opendata.parquet", index=False)

print("저장 완료: seoul_apt_trade_2022_opendata.csv / parquet")
df.head()


✅ 총 거래 건수: 823888
✅ 컬럼 수: 22
❌ 실패 건: 0
✅ 저장 완료: seoul_apt_trade_2022_opendata.csv / parquet


,aptDong,aptNm,buildYear,buyerGbn,cdealDay,cdealType,dealAmount,dealDay,dealMonth,dealYear,...,excluUseAr,floor,jibun,landLeaseholdGbn,rgstDate,sggCd,slerGbn,umdNm,구,계약년월
0,None,개포주공6단지,1983,None,None,None,"280,000",15,1,2022,...,83.21,8,185,N,None,11680,None,개포동,강남구,202201
1,None,개나리래미안,2006,None,None,None,"315,000",26,1,2022,...,129.8,18,754,N,None,11680,None,역삼동,강남구,202201
2,None,강남자곡 힐스테이트,2015,None,None,None,"103,000",8,1,2022,...,51.96,7,619,N,None,11680,None,자곡동,강남구,202201
3,None,우정에쉐르멤버스,2004,None,None,None,"58,000",7,1,2022,...,36.19,9,708-31,N,None,11680,None,역삼동,강남구,202201
4,None,삼성리치빌1,2003,None,None,None,"149,000",28,1,2022,...,84.98,4,52-9,N,None,11680,None,삼성동,강남구,202201


In [25]:
import pandas as pd

#  1) 파일 불러오기 (너가 저장한 파일명으로 수정)
df = pd.read_csv("seoul_apt_trade_2022_opendata.csv")

#  2) 날짜열 만들기: dealYear+dealMonth+dealDay → dealDate(YYYY-MM-DD)
df["dealDate"] = pd.to_datetime(
    df["dealYear"].astype(str) + "-" +
    df["dealMonth"].astype(str).str.zfill(2) + "-" +
    df["dealDay"].astype(str).str.zfill(2),
    errors="coerce"
).dt.strftime("%Y-%m-%d")

#  3) 주소열 만들기: 구 + umdNm + jibun → fullAddress
df["fullAddress"] = (
    df["구"].astype(str).str.strip() + " " +
    df["umdNm"].astype(str).str.strip() + " " +
    df["jibun"].astype(str).str.strip()
)

#  4) 필요하면 원래 열들 삭제(선택)
df.drop(columns=["dealYear","dealMonth","dealDay","sggCd","buyerGbn","slerGbn","landLeaseholdGbn","rgstDate","estateAgentSggNm","dealingGbn"], inplace=True)

#  5) 엑셀로 저장
df.to_excel("seoul_apt_trade_2022_preprocessed.xlsx", index=False)

print("저장 완료: seoul_apt_trade_2022_preprocessed.xlsx")
df[["dealDate", "fullAddress"]].head()


저장 완료: seoul_apt_trade_2022_preprocessed.xlsx


,dealDate,fullAddress
0,2022-01-15,강남구 개포동 185
1,2022-01-26,강남구 역삼동 754
2,2022-01-08,강남구 자곡동 619
3,2022-01-07,강남구 역삼동 708-31
4,2022-01-28,강남구 삼성동 52-9
